In [1]:
print("Hello World 1239")

Hello World 1239


In [2]:
import sys
print(sys.executable)

d:\Development\llm-zoomcamp\llm-zoomcamp-firstdraft\.venv\Scripts\python.exe


In [3]:
from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI
import os
open_client = OpenAI(
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1"
    )

In [ ]:
def llm(prompt):
    response = open_client.responses.create(
        model="google/gemini-2.5-flash",
        #model="google/gemma-4-31b-it:free",
        input=prompt,
        max_output_tokens=128
    )
    return response.output_text

question = "I just discovered the course can I join now?"
answer = llm(question)
print(answer)

#print(llm("Hey, tell me a joke."))

RateLimitError: Error code: 429 - {'error': {'message': 'Provider returned error', 'code': 429, 'metadata': {'raw': 'google/gemma-4-31b-it:free is temporarily rate-limited upstream. Please retry shortly, or add your own key to accumulate your rate limits: https://openrouter.ai/settings/integrations', 'provider_name': 'Google AI Studio', 'is_byok': False, 'previous_errors': [{'code': 429, 'message': 'Provider returned error', 'provider_name': 'OpenInference', 'raw': 'google/gemma-4-31b-it:free is temporarily rate-limited upstream. Please retry shortly, or add your own key to accumulate your rate limits: https://openrouter.ai/settings/integrations'}]}}, 'user_id': 'user_3Eixsj8xd6q7hGBZZ9pi59L7qAP'}

In [ ]:
#03-rag.md
context = """
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs. Students participate via YouTube Live and submit questions to Slido.

Cloud alternatives with GPU
Check the quota and reset cycle carefully. Potential options include Google Colab, Kaggle, Databricks.
"""

prompt = f"""
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."

Question:
{question}

Context:
{context}
"""
print(prompt)
answer = llm(prompt)
print(answer)


In [6]:
#04-dataset
import requests

docs_url = "https://datatalks.club/faq/json/courses.json"
#print(requests.get(docs_url))
response = requests.get(docs_url)
#print(response.json())
#print (type(response.json()))
courses_raw = response.json()
print(courses_raw[0])
#print(type(courses_raw[0]))

{'course': 'data-engineering-zoomcamp', 'course_name': 'Data Engineering Zoomcamp', 'path': '/json/data-engineering-zoomcamp.json', 'questions_count': 404}


In [7]:
import requests

def load_faq_data():
    docs_url = "https://datatalks.club/faq/json/courses.json"
    response = requests.get(docs_url)
    courses_raw = response.json()

    documents = []
    url_prefix = "https://datatalks.club/faq"
    for course in courses_raw:
        course_url = f"""{url_prefix}{course["path"]}"""
        course_response = requests.get(course_url)
        course_response.raise_for_status()
        course_data = course_response.json()
        documents.extend(course_data)
    return documents

full_documents = load_faq_data()

print(type(full_documents))
print (len(full_documents))


<class 'list'>
1368


In [19]:
documents[2]

NameError: name 'documents' is not defined

In [ ]:
documents[1100]

{'id': '841966c903',
 'course': 'mlops-zoomcamp',
 'section': 'Module 3: Orchestration',
 'question': 'Where is the FAQ for Prefect questions?',
 'answer': '[Here](https://docs.google.com/document/d/1Nyktf7WoRec5lDUBREXL5zLI1Edbw9_R8e45fDn4KB8/edit?usp=sharing)'}

In [8]:
#05-search
from minsearch import Index

def build_index(documents:list):
    index = Index(
        text_fields = ["question", "section", "answer"],
        keyword_fields = ["course"]
    )

    index.fit(documents)
    return index
index = build_index(full_documents)
print(type(index))

<class 'minsearch.minsearch.Index'>


In [ ]:
print(Index.__module__)
print(Index.__name__)


minsearch.minsearch
Index


In [ ]:
question = "I just discovered the course. Can I join now?"
question = "When baby will born, will it be a girl or a boy"

search_results = index.search(
    question,
    boost_dict = {"question":2.0, "section":0.5},
    filter_dict = {"course": "llm-zoomcamp"},
    num_results = 5
)

search_results

[{'id': 'bd31146b0e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'When will the course be offered next?',
  'answer': 'Summer 2027.'},
 {'id': 'eae0fb50aa',
  'course': 'llm-zoomcamp',
  'section': 'Capstone Project',
  'question': 'When and how will we be assigned projects for review/grading?',
  'answer': 'After the submission deadline has passed, an Excel sheet will be shared with 3 projects being assigned to each participant.'},
 {'id': '651ba06b34',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Will the name I put in the certificate field be shown publicly online or shared with third parties?',
  'answer': "No. The certificate name only appears on your certificate — it isn't published online or shared with third parties. The public leaderboard uses your separate display name instead, which is a random nickname (like “Lucid Elbakyan”) and anonymous by default.\n\nYou can set both in “Edit 

In [ ]:
print([doc["question"] for doc in search_results])
#[doc["question"] for doc in search_results]
print(type([doc["question"] for doc in search_results]))

print({doc["question"] for doc in search_results})
print(type({doc["question"] for doc in search_results}))

['When will the course be offered next?', 'When and how will we be assigned projects for review/grading?', 'Will the name I put in the certificate field be shown publicly online or shared with third parties?', 'OpenAI: How much will I have to spend to use the Open AI API?', 'How is my capstone project going to be evaluated?']
<class 'list'>
{'When and how will we be assigned projects for review/grading?', 'OpenAI: How much will I have to spend to use the Open AI API?', 'When will the course be offered next?', 'Will the name I put in the certificate field be shown publicly online or shared with third parties?', 'How is my capstone project going to be evaluated?'}
<class 'set'>


In [9]:
def search(question:str,course:str="llm-zoomcamp") -> list:
    boost_dict = {"question":2.0,"section":0.5}
    filter_dict = {"course":course}

    return index.search(
        question,
        boost_dict = boost_dict,
        filter_dict = filter_dict,
        num_results=5
    )
question = "I just discovered the course. Can I join now?"

search_results = search(question)
print(type(search_results))
#search_results

<class 'list'>


In [10]:
#06-building-prompt
INSTRUCTIONS = """
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
"""
USER_PROMPT_TEMPLATE = """
Question:
{question}

Context:
{context}
"""

def build_context(search_results):
    lines = []
    for doc in search_results:
        lines.append(doc["section"])
        lines.append("Q. "+doc["question"])
        lines.append("A. "+doc["answer"])
        lines.append("")
    
    return "\n".join(lines).strip()

def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(
        question = question,
        context = context
    )
    return prompt.strip()

prompt = build_prompt(question,search_results)
print(prompt)
    

Question:
I just discovered the course. Can I join now?

Context:
General Course-Related Questions
Q. I just discovered the course. Can I still join?
A. Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

General Course-Related Questions
Q. Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
A. You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

General Course-Related Questions
Q. Certificate: Can I follow the course in a self-paced mode and get a certificate?
A. No, you can only get a certificate if you finish the course with a "live" cohort.

We don't award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project

In [ ]:
#07-llm
from pprint import pprint

#print(prompt)

response = open_client.responses.create(
     #model="google/gemini-2.5-flash",
        model="google/gemma-4-31b-it:free",
        input=INSTRUCTIONS+prompt,
        max_output_tokens=128    
)
print(response.output_text)
#print(response.id)
#print(response.text)
#pprint(response.model_dump())
#help(response)
#print(dir(response))
#response.output[0].content[0].text
#print(response.usage)

input_price=0.75/1_000_000
output_price=4.50//1_000_000

total_cost = (response.usage.input_tokens*input_price + response.usage.input_tokens*output_price)
print(total_cost)

Yes, you can still join. However, if you want to receive a certificate, you must submit your project while submissions are still being accepted.
0.0005902500000000001


In [ ]:
message_history = [{"role":"developer","content":INSTRUCTIONS},
                   {"role":"user","content":prompt}]
response = open_client.responses.create(
     #model="google/gemini-2.5-flash",
        model="google/gemma-4-31b-it:free",
        input=message_history,
        max_output_tokens=128    
)

response.output_text

'Yes, you can still join. However, if you want to receive a certificate, you must submit your project while submissions are still being accepted.'

In [11]:
def llm(instructions:str,prompt:str,model:str="google/gemini-2.5-flash") -> str:
    message_history = [
        {"role":"developer", "content":INSTRUCTIONS},
        {"role":"user", "content":prompt}
        ]

    response = open_client.responses.create( 
        model=model, 
        input=message_history,
        max_output_tokens=128
        )
    return response.output_text

def rag(question:str) -> str:
    #docs = load_faq_data()
    #index = build_index(docs)
    search_results = search(question)
    prompt = build_prompt(question,search_results)
    answer = llm(INSTRUCTIONS,prompt)
    return answer

print(rag("I just discovered the course. Can I join now?"))

Yes, you can join now. However, if you want to receive a certificate, you need to submit your project while submissions are still being accepted. You can start whenever you want, as the videos and GitHub materials are available.


In [42]:
'##########           #....... Section 3.2.1 Issue #32 .......'.strip("#")

'           #....... Section 3.2.1 Issue #32 .......'